# 2.1 Using Tools

*Using Microsoft Semantic Kernel (Agent Framework)*

By binding tools or functions to your LLM, the model gets information about those tools, and it will tell you that it wants to execute a tool. Notice the LLM is not going to execute the tool, it's the server where the process is running. That same process will also return the result of those tools back to the LLM.

This notebook demonstrates the same concepts as the LangChain version, but using Microsoft Semantic Kernel.

In [ ]:
import os
from typing import Annotated
from dotenv import load_dotenv
import semantic_kernel as sk
from semantic_kernel.connectors.ai.open_ai import AzureChatCompletion
from semantic_kernel.contents import ChatHistory
from semantic_kernel.functions import kernel_function

Define the kernel and model

In [ ]:
load_dotenv()

# Initialize the kernel
kernel = sk.Kernel()

# Add Azure OpenAI Chat Completion service
service_id = "chat-gpt"
kernel.add_service(
    AzureChatCompletion(
        service_id=service_id,
        deployment_name="gpt-4o-mini",
        endpoint=os.getenv("AZURE_OPENAI_ENDPOINT"),
        api_key=os.getenv("AZURE_OPENAI_API_KEY"),
    )
)

Define the tool/action the LLM will have access to

In [ ]:
class WeatherPlugin:
    """
    A plugin to get weather information.
    """
    
    @kernel_function(
        name="get_weather",
        description="Get the current weather for a specified location"
    )
    def get_weather(
        self,
        location: Annotated[str, "The name of the city. Must be one of: Chicago, New York, or Los Angeles"]
    ) -> str:
        """
        Get the current weather for a specified location.
        """
        weather_data = {
            "New York": "Sunny, 25°C",
            "Los Angeles": "Cloudy, 22°C",
            "Chicago": "Rainy, 18°C",
        }
        return weather_data.get(location, "Weather data not available for this location.")

# Add the plugin to the kernel
kernel.add_plugin(WeatherPlugin(), plugin_name="weather")

Set up the chat history with system and user messages

In [ ]:
chat_history = ChatHistory()
chat_history.add_system_message(
    "You are a helpful assistant that can provide weather information for specific cities."
)
chat_history.add_user_message("What is the weather like in NYC?")

Call the LLM with automatic tool execution enabled

In [ ]:
from semantic_kernel.connectors.ai.open_ai.prompt_execution_settings.azure_chat_prompt_execution_settings import (
    AzureChatPromptExecutionSettings,
)
from semantic_kernel.connectors.ai.function_choice_behavior import FunctionChoiceBehavior

# Configure execution settings to enable automatic function calling
execution_settings = AzureChatPromptExecutionSettings(
    service_id=service_id,
    function_choice_behavior=FunctionChoiceBehavior.Auto(),
)

# Get the chat completion service
chat_service = kernel.get_service(service_id)

# Get the response with automatic function calling
response = await chat_service.get_chat_message_content(
    chat_history=chat_history,
    settings=execution_settings,
    kernel=kernel,
)

chat_history.add_message(response)

Print the result

In [ ]:
print("Final response:")
print(response)

print("\nFull chat history:")
for message in chat_history.messages:
    print(f"{message.role}: {message.content}")

Notice how Semantic Kernel automatically handles the tool calls and returns the final response.

**NOTE:** Usage of tools means 2 LLM calls, one to know what tool is necessary and second one containing the result of the tool call